In [ ]:
import pandas as pd

file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset loaded successfully!
Shape: (6362620, 11)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [30]:
print("Minimum step:", df["step"].min())
print("Maximum step:", df["step"].max())
print("Unique steps:", df["step"].nunique())

Minimum step: 1
Maximum step: 743
Unique steps: 743


In [31]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 6362620
Number of columns: 11


In [32]:
print(df.isnull().sum())

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [37]:
print( " Duplicate rows : ", df.duplicated().sum())

 Duplicate rows :  0


In [ ]:
print(" fraud calls:", df ["isFraud"] . value_counts()) #this generates total % of fraud and non fraud calls ..... result is 0.13% fraud and 99.87% not fraud huge difference.

 fraud calls: isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [ ]:
print ( df ["type"] .value_counts ())

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [ ]:
print(df.groupby("type")["isFraud"].sum()) #This will group the transactions by type and count the fraudulent transactions (isFraud = 1) within each type. almost cash_out and  transfer are  fraud .

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64


In [ ]:
print(df.groupby("type")["isFraud"].mean() * 100)  #checking the fraud rate within each transaction type, rather than just the fraud count.
#About 0.77% of TRANSFER transactions are fraudulent.
#About 0.18% of CASH_OUT transactions are fraudulent.
#0.77% is still less than 1%.
#So even among TRANSFER transactions, the vast majority are legitimate.
#we shouldn't simply create a rule like "TRANSFER = fraud."
#Our model needs to consider other information too, such as:
#transaction amount
#ender's balance
#receiver's balance
#transaction time (step)
#possibly account-related features

type
CASH_IN     0.000000
CASH_OUT    0.183955
DEBIT       0.000000
PAYMENT     0.000000
TRANSFER    0.768799
Name: isFraud, dtype: float64


In [ ]:
#Let's check whether fraudulent transactions tend to have larger amounts.

print(df.groupby("isFraud")["amount"].describe ())

#remeber ---- method/function → () , column selection → []
#Group transactions by whether they are fraud (0 or 1) 
# select their amount → describe the amount distribution.

#from belows output....
#| Statistic       |      Normal |          Fraud |
#| --------------- | ----------: | -------------: |
#| Count           |   6,354,407 |          8,213 |
#| Mean            |    ₹178,197 | **₹1,467,967** |
#| Median (50%)    |     ₹74,685 |   **₹441,423** |
#| 75th percentile |    ₹208,365 | **₹1,517,771** |
#| Maximum         | ₹92,445,517 |    ₹10,000,000 |

#The important observation:
 #The average fraudulent transaction amount is much higher
#But notice the maximum:
#Normal → ₹92.4 million
#Fraud  → ₹10 million
#So amount alone cannot determine fraud. There are legitimate (non fraud) transactions with very large amounts.
#what is e+06,e+05??
#is scientific notation.
#It means:
#1.467967 × 10⁶= 1,467,967
#Similarly:1.781970e+05= 178,197 ......

## our inspection is going to reveal some useful pattern.
#Transaction type ───────┐
#                        │
#Transaction amount ─────┼──→ Fraud prediction
#                        │
#Balance information ────┘
#But we're not deciding which columns to keep yet. 
#First we'll inspect the dataset thoroughly, then deal with leakage and feature selection.



             count          mean           std   min         25%        50%  \
isFraud                                                                       
0        6354407.0  1.781970e+05  5.962370e+05  0.01   13368.395   74684.72   
1           8213.0  1.467967e+06  2.404253e+06  0.00  127091.330  441423.44   

                75%          max  
isFraud                           
0         208364.76  92445516.64  
1        1517771.48  10000000.00  


In [ ]:
#Let's examine whether fraudulent transactions behave differently in terms of sender balances.
print(df.groupby("isFraud")["oldbalanceOrg"].describe())

#output says that:
#| Statistic |      Normal |       Fraud |
#| --------- | ----------: | ----------: |
#| Count     |   6,354,407 |       8,213 |
#| Mean      |    ₹832,829 |  ₹1,649,668 |
#| Median    |     ₹14,069 |    ₹438,983 |
#| 75%       |    ₹106,970 |  ₹1,517,771 |
#| Maximum   | ₹43,818,855 | ₹59,585,040 |

#The interesting part

#The median is particularly informative:

#Normal transactions → ₹14,069
#Fraudulent          → ₹438,983

#That's a very large difference.
#So fraudulent transactions in this dataset tend to involve accounts with substantially higher sender balances before the transaction.
#But again, this doesn't mean:
#"High balance = fraud."
#There are plenty of NON FRAUD transactions with high balances..




             count          mean           std  min        25%        50%  \
isFraud                                                                     
0        6354407.0  8.328287e+05  2.887144e+06  0.0       0.00   14069.00   
1           8213.0  1.649668e+06  3.547719e+06  0.0  125822.44  438983.45   

                75%          max  
isFraud                           
0         106969.50  43818855.30  
1        1517771.48  59585040.37  


In [ ]:
#The next thing we should investigate is what happened to the sender's balance after the transaction.
print(df.groupby("isFraud")["newbalanceOrig"].describe())

#| Statistic |      Normal |       Fraud |
#| --------- | ----------: | ----------: |
#| Mean      |    ₹855,970 |    ₹192,393 |
#| Median    |          ₹0 |          ₹0 |
#| 75%       |    ₹144,731 |      **₹0** |
#| Maximum   | ₹43,686,616 | ₹49,585,040 |


#The interesting point is the 75th percentile:
#Normal: about ₹1.45 lakh
#Fraud: ₹0
#That means at least 75% of the fraudulent transactions have newbalanceOrig of ₹0 in this dataset.
#But here's an important warning ⚠️
#Remember our earlier discussion about data leakage.
#newbalanceOrig is the sender's balance after the transaction. 
# If our real-world fraud detector is supposed to make a decision before or during the transaction, we need to ask:
#Would this information actually be available at prediction time?
#If the answer is no, using it could make our model look artificially accurate.
#So we're not dropping it yet. We're inspecting it first and
# will make the feature-selection decision after understanding the dataset.

             count           mean           std  min  25%  50%        75%  \
isFraud                                                                     
0        6354407.0  855970.228109  2.924987e+06  0.0  0.0  0.0  144730.74   
1           8213.0  192392.631836  1.965666e+06  0.0  0.0  0.0       0.00   

                 max  
isFraud               
0        43686616.33  
1        49585040.37  


In [ ]:
#let's inspect the receiver's balance before the transaction:
print(df.groupby("isFraud")["oldbalanceDest"].describe())

#Compare normal vs fraud
#| Statistic |       Normal |        Fraud |
#| --------- | -----------: | -----------: |
#| Mean      |  ₹11.01 lakh |   ₹5.44 lakh |
#| Median    |   ₹1.33 lakh |       **₹0** |
##| 75%       |   ₹9.44 lakh |   ₹1.48 lakh |
#| Maximum   | ₹35.60 crore | ₹23.62 crore |


##Important observation
#The biggest difference is again around the median:
#Normal transactions → ₹1,33,312
#Fraudulent          → ₹0
#For fraudulent transactions, at least 50% have oldbalanceDest = 0.
#So receiver-side balance also appears to contain useful information.


             count          mean           std  min  25%       50%        75%  \
isFraud                                                                         
0        6354407.0  1.101421e+06  3.399202e+06  0.0  0.0  133311.8  944144.58   
1           8213.0  5.442496e+05  3.336421e+06  0.0  0.0       0.0  147828.66   

                  max  
isFraud                
0        3.560159e+08  
1        2.362305e+08  


In [ ]:
#Let's inspect the receiver's balance after the transaction:
print(df.groupby("isFraud")["newbalanceDest"].describe())

#| Statistic |       Normal |        Fraud |
#| --------- | -----------: | -----------: |
#| Mean      |  ₹12.25 lakh |  ₹12.80 lakh |
#| Median    |   ₹2.15 lakh |   **₹4,676** |
#| 75%       |  ₹11.12 lakh |  ₹10.59 lakh |
#| Maximum   | ₹35.62 crore | ₹23.67 crore |

#The mean isn't very different, but the median is dramatically different:
#Normal → ₹2,14,882
#Fraud  → ₹4,676
#So the receiver's post-transaction balance also shows a different distribution for fraud.




             count          mean           std  min  25%        50%  \
isFraud                                                               
0        6354407.0  1.224926e+06  3.673816e+06  0.0  0.0  214881.70   
1           8213.0  1.279708e+06  3.908817e+06  0.0  0.0    4676.42   

                 75%           max  
isFraud                             
0        1111975.345  3.561793e+08  
1        1058725.220  2.367265e+08  


In [ ]:
## what we have observed till now ????? 
#Feature             What we observed
#────────────────────────────────────────────
#amount              Fraud amounts generally higher
#oldbalanceOrg       Fraud generally higher
#newbalanceOrig      Fraud often ends at 0
#oldbalanceDest      Fraud often starts at 0
#newbalanceDest      Fraud median is much lower


#Now comes an important part
#We should not immediately put all these columns into the model.
#The four balance columns describe what happened around/after the transaction, 
# and some may contain information that wouldn't be available at the moment 
# a real fraud-detection system needs to make its decision.
#So our next step should be:
#Understand the PaySim transaction process and identify potential data leakage before preprocessing.
#After that we'll decide which features are appropriate for our model 
# rather than blindly keeping or dropping columns.


In [ ]:
'''| Feature          | When it exists     | Initial assessment   |
   | ---------------- | ------------------ | -------------------- |
   | `oldbalanceOrg`  | Before transaction | ✅ Potentially usable |
   | `oldbalanceDest` | Before transaction | ✅ Potentially usable |
   | `newbalanceOrig` | After transaction  | ⚠️ Potential leakage |
   | `newbalanceDest` | After transaction  | ⚠️ Potential leakage |

we create a realistic pre-transaction model and therefore exclude the post-transaction balances.
We can still keep them in the raw dataset for analysis. '''
